In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC

from bs4 import BeautifulSoup
import re


sns.set_style('darkgrid')

In [2]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
df.sample(5)

,review,sentiment
28428,"First, they ruin it with the uniquely bad anim...",negative
3089,This film shows up on the premium cable channe...,positive
40044,This film was not only one of John Ford's own ...,positive
44935,This movie just arrived to Mexico and since I ...,negative
28487,I've read plenty of Jane Austen in my time and...,negative


In [5]:
df.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [6]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [7]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [8]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

In [9]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [10]:
train_x.shape

(6700,)

In [11]:
train_x_vector.shape

(6700, 44107)

In [12]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

In [13]:
pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out())

,00,000,007,00am,00s,01,01pm,02,04,05,...,émigré,émigrés,était,étc,être,ísnt,île,önsjön,über,überwoman
6746,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
54,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8499,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7869,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3725,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
358,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
761,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1685,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [15]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [16]:
primera_resenia

00           0
000          0
007          0
00am         0
00s          0
            ..
ísnt         0
île          0
önsjön       0
über         0
überwoman    0
Name: 6746, Length: 44107, dtype: Sparse[float64, 0]

In [17]:
primera_resenia[primera_resenia != 0]

13               0.45849
17               0.12824
actual          0.091601
actually        0.061461
age             0.088765
allowing         0.12824
beware          0.143046
br              0.124945
bunch           0.093128
deplorable      0.168137
disgusting      0.237945
entertaining    0.081088
expectations    0.103149
great           0.048868
happened        0.176853
hopes           0.114028
humor           0.084465
humorous        0.116516
including       0.087603
joked           0.168137
just            0.037675
know            0.053984
let              0.07195
like            0.036454
mature          0.126021
movie           0.276309
movies            0.0522
old             0.123513
pedophilia      0.172712
pg              0.260154
rated           0.206643
recommend       0.076198
references      0.226901
rent            0.093234
revolting       0.151971
safe            0.119732
sexual          0.098677
shouldn         0.108203
sister          0.095008
small           0.078916


In [18]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [19]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

['positive']
['positive']
['negative']


In [20]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [21]:
print(svc.score(test_x_vector, test_y))

0.8706060606060606


In [22]:
f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87400413, 0.86701962])

In [23]:
print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [24]:
conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1481,  197],
       [ 230, 1392]])

In [25]:
# 1. Importar el modelo
from sklearn.linear_model import LogisticRegression

# 2. Inicializar y entrenar la Regresión Logística
lr = LogisticRegression(random_state=42)
lr.fit(train_x_vector, train_y)

# 3. Realizar predicciones en el conjunto de prueba
predicciones_lr = lr.predict(test_x_vector)


/Users/veshar/procesamiento_ln/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/veshar/procesamiento_ln/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/veshar/procesamiento_ln/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


In [26]:
# 4. Calcular el Accuracy (Score general)
print("Accuracy de Logistic Regression:")
print(lr.score(test_x_vector, test_y))
print("\n" + "="*50 + "\n")

# 5. Reporte de clasificación (Precisión, Recall, F1-Score)
print("Reporte de Clasificación:")
print(classification_report(test_y, predicciones_lr, labels=['positive', 'negative']))
print("\n" + "="*50 + "\n")

# 6. Matriz de confusión
print("Matriz de Confusión:")
conf_mat_lr = confusion_matrix(test_y, predicciones_lr, labels=['positive', 'negative'])
print(conf_mat_lr)


Accuracy de Logistic Regression:
0.8718181818181818


Reporte de Clasificación:
              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



Matriz de Confusión:
[[1479  199]
 [ 224 1398]]


In [27]:
# 7. Pruebas individuales con frases personalizadas
print("Predicción ['A good movie']:", lr.predict(tfidf.transform(['A good movie'])))
print("Predicción ['An excellent movie']:", lr.predict(tfidf.transform(['An excellent movie'])))
print("Predicción ['I did not like this movie at all I gave this movie away']:", 
      lr.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))


Predicción ['A good movie']: ['positive']
Predicción ['An excellent movie']: ['positive']
Predicción ['I did not like this movie at all I gave this movie away']: ['negative']


In [28]:
# 1. Importar el modelo GaussianNB
from sklearn.naive_bayes import GaussianNB

# 2. Convertir las matrices de texto vectorizado a formato denso (.toarray())
train_x_dense = train_x_vector.toarray()
test_x_dense = test_x_vector.toarray()

# 3. Inicializar y entrenar el modelo con los datos densos
gnb = GaussianNB()
gnb.fit(train_x_dense, train_y)

# 4. Realizar predicciones en el conjunto de prueba
predicciones_gnb = gnb.predict(test_x_dense)


In [29]:
# 5. Calcular el Accuracy (Score general)
print("Accuracy de GaussianNB:")
print(gnb.score(test_x_dense, test_y))
print("\n" + "="*50 + "\n")

# 6. Reporte de clasificación (Precisión, Recall, F1-Score)
print("Reporte de Clasificación:")
print(classification_report(test_y, predicciones_gnb, labels=['positive', 'negative']))
print("\n" + "="*50 + "\n")

# 7. Matriz de confusión
print("Matriz de Confusión:")
conf_mat_gnb = confusion_matrix(test_y, predicciones_gnb, labels=['positive', 'negative'])
print(conf_mat_gnb)


Accuracy de GaussianNB:
0.6427272727272727


Reporte de Clasificación:
              precision    recall  f1-score   support

    positive       0.66      0.62      0.64      1678
    negative       0.63      0.67      0.65      1622

    accuracy                           0.64      3300
   macro avg       0.64      0.64      0.64      3300
weighted avg       0.64      0.64      0.64      3300



Matriz de Confusión:
[[1033  645]
 [ 534 1088]]


In [30]:
# 8. Pruebas individuales (se debe usar .toarray() en el vector transformado)
print("Predicción ['A good movie']:", gnb.predict(tfidf.transform(['A good movie']).toarray()))
print("Predicción ['An excellent movie']:", gnb.predict(tfidf.transform(['An excellent movie']).toarray()))
print("Predicción ['I did not like this movie at all I gave this movie away']:", 
      gnb.predict(tfidf.transform(['I did not like this movie at all I gave this movie away']).toarray()))


Predicción ['A good movie']: ['negative']
Predicción ['An excellent movie']: ['negative']
Predicción ['I did not like this movie at all I gave this movie away']: ['negative']


In [31]:
# 1. Importar el modelo DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier

# 2. Inicializar y entrenar el Árbol de Decisión
# Usamos random_state=42 para que el árbol se construya igual cada vez que corras el código
dtc = DecisionTreeClassifier(random_state=42)
dtc.fit(train_x_vector, train_y)

# 3. Realizar predicciones en el conjunto de prueba
predicciones_dtc = dtc.predict(test_x_vector)


In [32]:
# 4. Calcular el Accuracy (Score general)
print("Accuracy de Decision Tree Classifier:")
print(dtc.score(test_x_vector, test_y))
print("\n" + "="*50 + "\n")

# 5. Reporte de clasificación (Precisión, Recall, F1-Score)
print("Reporte de Clasificación:")
print(classification_report(test_y, predicciones_dtc, labels=['positive', 'negative']))
print("\n" + "="*50 + "\n")

# 6. Matriz de confusión
print("Matriz de Confusión:")
conf_mat_dtc = confusion_matrix(test_y, predicciones_dtc, labels=['positive', 'negative'])
print(conf_mat_dtc)


Accuracy de Decision Tree Classifier:
0.7163636363636363


Reporte de Clasificación:
              precision    recall  f1-score   support

    positive       0.73      0.71      0.72      1678
    negative       0.71      0.72      0.71      1622

    accuracy                           0.72      3300
   macro avg       0.72      0.72      0.72      3300
weighted avg       0.72      0.72      0.72      3300



Matriz de Confusión:
[[1192  486]
 [ 450 1172]]


In [33]:
# 7. Pruebas individuales con frases personalizadas
print("Predicción ['A good movie']:", dtc.predict(tfidf.transform(['A good movie'])))
print("Predicción ['An excellent movie']:", dtc.predict(tfidf.transform(['An excellent movie'])))
print("Predicción ['I did not like this movie at all I gave this movie away']:", 
      dtc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))


Predicción ['A good movie']: ['positive']
Predicción ['An excellent movie']: ['positive']
Predicción ['I did not like this movie at all I gave this movie away']: ['positive']
